# Practice 06B — LLM Internals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/06b-llm-internals.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/06b-llm-internals.ipynb)

Companion drills for **Phase 6B** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with the type-your-own-corpus [BPE explainer](https://learn-by-visuallization.org/illustrated/6b-llm-indepth/tokenization-bpe.html)
and [LLM decoding](https://learn-by-visuallization.org/illustrated/6b-llm-indepth/llm-decoding.html).

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup, no GPU needed).

In [ ]:
NEEDED = [("numpy", "numpy")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
rng = np.random.default_rng(0)


### Exercise 1 — one BPE merge
Implement `merge_pair(symbols, a, b)`: scan the symbol list left-to-right and fuse every adjacent
`a, b` into `a+b` (non-overlapping, single pass) — the atom of the BPE explainer.

*Hint: walk with an index; on a match append `a+b` and skip 2, else append and skip 1.*

In [ ]:
def merge_pair(symbols, a, b):
    # TODO
    ...


In [ ]:
check("basic merge", lambda: merge_pair(list("newest"), "e", "s") == ["n", "e", "w", "es", "t"],
      "only ADJACENT e,s fuse — the first e stays alone")
check("repeated pairs all merge in one pass",
      lambda: merge_pair(list("aaaa"), "a", "a") == ["aa", "aa"],
      "after fusing positions 0-1, continue from position 2 (no overlap)")

### Exercise 2 — BPE training on your own text
Using `merge_pair`, implement `train_bpe(text, n_merges)`: split each word into chars, repeatedly
find the most frequent adjacent pair across all words (ties: lexicographic smallest), merge it
everywhere, record the rule. Return the list of rules.

*Hint: count pairs word by word; `max(counts, key=lambda k: (counts[k], k))` breaks ties wrong — sort
by (-count, pair) instead.*

In [ ]:
def train_bpe(text, n_merges=5):
    words = [list(w) for w in text.lower().split()]
    rules = []
    # TODO
    return rules


In [ ]:
_r = train_bpe("low low low lower lower newest newest newest newest widest", 3)
check("3 rules learned", lambda: len(_r) == 3, "one rule per merge round")
check("first merge is the most frequent pair: w+e (x6)",
      lambda: tuple(_r[0])[:2] == ("w", "e"),
      "count across ALL words: w-e appears in newest(x4) + lower(x2) = 6 - more than es (5) or lo (5)")

### Exercise 3 — temperature
Implement `softmax_T(logits, T)`. Then feel the knob: at T→0 the distribution collapses onto the
argmax (greedy); at high T it flattens toward uniform (creative/chaotic).

*Hint: softmax(logits / T), with the subtract-max trick for stability.*

In [ ]:
logits = np.array([2.0, 1.0, 0.2, -1.0])

def softmax_T(x, T=1.0):
    # TODO
    ...


In [ ]:
check("probabilities sum to 1", lambda: abs(softmax_T(logits, 1.0).sum() - 1) < 1e-9, "normalize")
check("low T sharpens toward greedy",
      lambda: softmax_T(logits, 0.1)[0] > 0.99, "dividing by 0.1 multiplies logit gaps ×10")
check("high T flattens toward uniform",
      lambda: abs(softmax_T(logits, 100.0)[0] - 0.25) < 0.02, "gaps shrink ×0.01 — almost a fair coin")

### Exercise 4 — top-k sampling
Implement `top_k(probs, k)`: zero out everything but the k highest probabilities and renormalize.
This is the guard that stops the model from ever picking the long tail of nonsense tokens.

*Hint: `np.argsort(probs)[:-k]` are the indices to zero.*

In [ ]:
def top_k(probs, k):
    # TODO: return the truncated + renormalized distribution
    ...


In [ ]:
_pr = np.array([0.5, 0.3, 0.15, 0.05])
check("only k tokens survive", lambda: (top_k(_pr, 2) > 0).sum() == 2, "zero the rest")
check("survivors are renormalized",
      lambda: abs(top_k(_pr, 2).sum() - 1) < 1e-9 and abs(top_k(_pr, 2)[0] - 0.625) < 1e-9,
      "0.5/(0.5+0.3) = 0.625 — relative odds among survivors are preserved")

### Stretch goal — nucleus (top-p)
Implement top-p: keep the smallest set of tokens whose cumulative probability ≥ p. Compare with
top-k on a flat vs peaked distribution — notice how top-p adapts while top-k cannot.

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")
